# Proyecto Final — IA Generativa
## Asistente Experto con Gemini, RAG y Agentes

**Alumno:** David Gómez Picazo
**Máster en Data Science**
**Módulo:** IA Generativa

## 1. Introducción y arquitectura del sistema

El dominio del agente experto es la **propia teoría del módulo de IA Generativa**. La base de conocimiento se construye a partir de los PDFs proporcionados en el módulo


## 2. Configuración del entorno

### 2.1. Instalación de dependencias

Las librerías necesarias se enumeran a continuación. La celda se mantiene comentada para evitar reinstalaciones; basta con ejecutarla la primera vez que se utiliza el notebook.

In [ ]:
# Dependencias del proyecto. Descomentar y ejecutar solo la primera vez.
# !pip install -q langchain langchain-core langchain-google-genai langgraph \
#                  langchain-community langchain-chroma chromadb pypdf \
#                  python-dotenv langchain-text-splitters \
#                  langchain-huggingface sentence-transformers ipywidgets

### 2.2. Carga de la API key de Gemini

Siguiendo las buenas prácticas , la clave de la API no se incluye en el código fuente. Se almacena en un fichero `API.env` y se carga con `python-dotenv`.

In [1]:
import os
from dotenv import load_dotenv

# Cargamos las variables de entorno definidas en el fichero local API.env
load_dotenv('./API.env')

# La clave se almacena bajo el nombre API_KEY1 en el fichero .env
GEMINI_API_KEY = os.getenv('API_KEY1')

# Los clientes oficiales de Google leen la clave desde GOOGLE_API_KEY
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

# Validación: si la clave no se ha podido cargar, abortamos la ejecución
if GEMINI_API_KEY:
    print(f'API Key cargada: {GEMINI_API_KEY[:10]}...')
else:
    raise ValueError('No se ha podido cargar la API Key. Revisa el fichero API.env')

API Key cargada: AIzaSyB9r6...


## 3. Construcción del corpus documental

### 3.1. Carga de los PDFs

El corpus se compone de los documentos de teoría del módulo, que los resumí en 4 páginas y se están ubicados en la carpeta `./PDFs/`.

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# El loader recorre la carpeta indicada y carga todos los PDFs disponibles.
# Cada página se convierte en un Document con su texto y su metadata.
loader = PyPDFDirectoryLoader('./PDFs/')

print('Cargando PDFs...')
documentos = loader.load()

# Extraemos las fuentes únicas para mostrar de qué ficheros proceden las páginas.
fuentes = set(d.metadata.get('source', '') for d in documentos)

print(f'Ficheros cargados : {len(fuentes)}')
print(f'Páginas en total  : {len(documentos)}')
print()
for f in sorted(fuentes):
    print(f'  - {os.path.basename(f)}')

Cargando PDFs...
Ficheros cargados : 4
Páginas en total  : 16

  - Ingenieria_de_Prompts.pdf
  - Introduccion_LLMs.pdf
  - Orquestacion_Despliegue_Agentes.pdf
  - RAG_langgraph.pdf


### 3.2. Segmentación en *chunks*

**Hiperparámetros elegidos:**

- `chunk_size = 700` caracteres: tamaño suficiente para que cada fragmento conserve sentido propio sin sobrecargar el contexto.
- `chunk_overlap = 200` caracteres: solapamiento entre fragmentos consecutivos, para no perder ideas que queden a caballo entre dos cortes.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter intenta primero cortar por dobles saltos de línea (separadores naturales de párrafo) y, si no es posible, va descendiendo hasta
# saltos simples, espacios y, en último caso, caracteres individuales.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    separators=['\n\n', '\n', ' ', '']
)

# Aplicamos el splitter a la lista completa de documentos.
fragmentos = splitter.split_documents(documentos)

print(f'Páginas originales   : {len(documentos)}')
print(f'Chunks generados     : {len(fragmentos)}')

Páginas originales   : 16
Chunks generados     : 48


## 4. Base de conocimiento vectorial

### 4.1. Modelo de embeddings

Convertimos cada fragmento de texto en un vector numérico (*embedding*).


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil

# Inicializamos el modelo de embeddings. La primera vez descarga los pesos a caché local.
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Parámetros de la base ChromaDB
CHROMA_PATH = './chroma_db'
COLECCION   = 'proyecto_ia_master'

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 4.2. Indexación con ChromaDB

Almacenamos cada fragmento con su embedding y su metadata en *ChromaDB*, para poder recuperar después tanto el texto como la procedencia.

In [5]:
# Si la carpeta existe pero la colección está vacía (por ejemplo, una ejecución
# previa abortada), la borramos para forzar una creación limpia.
if os.path.exists(CHROMA_PATH):
    vectorstore_temp = Chroma(
        collection_name=COLECCION,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )
    if vectorstore_temp._collection.count() == 0:
        del vectorstore_temp
        shutil.rmtree(CHROMA_PATH)

# Si la base ya existe y contiene datos, la reaprovechamos. Si no, la creamos
# a partir de los fragmentos generados en el paso anterior.
if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):
    print('Cargando base existente...')
    vectorstore = Chroma(
        collection_name=COLECCION,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )
else:
    print(f'Creando base con {len(fragmentos)} fragmentos...')
    vectorstore = Chroma.from_documents(
        documents=fragmentos,
        embedding=embeddings,
        collection_name=COLECCION,
        persist_directory=CHROMA_PATH
    )

print(f'Fragmentos indexados: {vectorstore._collection.count()}')

Creando base con 48 fragmentos...
Fragmentos indexados: 48


### 4.3. Validación del *retriever*

Antes de conectar la base vectorial al agente conviene comprobar que las consultas devuelven fragmentos coherentes. Si la recuperación falla, el agente tampoco podrá responder bien. Se busca por similitud del coseno y de devuelven los 3 fragmentos más próximos a la consulta

In [6]:
# Creamos el retriever a partir del vectorstore.
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# Lanzamos un par de consultas de prueba para verificar la calidad de la recuperación.
preguntas_test = [
    '¿Qué es RAG?',
    '¿Cómo funciona LangGraph?'
]

for pregunta in preguntas_test:
    resultados = retriever.invoke(pregunta)
    print(f'Pregunta: "{pregunta}"')
    for i, doc in enumerate(resultados, 1):
        fichero = os.path.basename(doc.metadata.get('source', '?'))
        pagina  = doc.metadata.get('page', '?')
        print(f'  [{i}] {fichero} | pág. {pagina}')
        print(f'      {doc.page_content[:150].strip()}...')
    print()

print('Retriever validado correctamente.')

Pregunta: "¿Qué es RAG?"
  [1] RAG_langgraph.pdf | pág. 0
      RAG y Agentes con LangGraph:
Arquitectura de Sistemas de IA Generativa
 Documento de referencia técnica en español — IA Generativa y Sistemas RAG
1. I...
  [2] Ingenieria_de_Prompts.pdf | pág. 3
      12. Gamificación y simulación
La gamificación transforma la interacción en escenarios o juegos para mejorar creatividad y
calidad.

Uso de roles y es...
  [3] RAG_langgraph.pdf | pág. 3
      claramente que no dispones de esa informacion en tu base de conocimiento y sugiere
donde podria encontrarse. Usa un tono tecnico pero claro. Incluye e...

Pregunta: "¿Cómo funciona LangGraph?"
  [1] Introduccion_LLMs.pdf | pág. 2
      6. Funcionamiento de los LLMs
Los LLMs generan texto token a token. En cada paso calculan la probabilidad del siguiente token
en función del contexto...
  [2] RAG_langgraph.pdf | pág. 0
      RAG y Agentes con LangGraph:
Arquitectura de Sistemas de IA Generativa
 Documento de referencia técnica en español

## 5. Diseño del agente

### 5.1. Herramienta RAG

Hasta aquí, buscaría siempre aunque la pregunta no lo necesite.Pero es mejor buscar solo cuándo es necesario, por ello se crea la herramienta que obligue a buscar cuándo se trata de temas realcionados con este agente

In [ ]:
from langchain_core.tools import tool


@tool
def buscar_en_base_conocimiento(consulta: str) -> str:
    '''Busca información relevante en los PDFs del curso de IA Generativa.

    Úsala cuando el usuario pregunte sobre:
    - RAG, Retrieval-Augmented Generation, recuperación de documentos
    - LangGraph, grafos de estado, agentes, patrón ReAct
    - Embeddings, vectores, similitud coseno, ChromaDB
    - LLMs, modelos de lenguaje, tokens
    - Prompt Engineering, few-shot, output parsers
    - Memoria en agentes, MemorySaver, checkpoints

    No la utilices para saludos ni para preguntas ajenas al contenido del curso.
    '''
    # Recuperamos los k=3 fragmentos más similares semánticamente a la consulta.
    docs = retriever.invoke(consulta)

    # Construimos un bloque de texto con cada fragmento y su procedencia.
    # El formato facilita que el LLM cite la fuente al responder.
    partes = []
    for doc in docs:
        fichero = os.path.basename(doc.metadata.get('source', '?'))
        pagina  = doc.metadata.get('page', '?')
        partes.append(f'[{fichero} | pág. {pagina}]\n{doc.page_content.strip()}')

    return '\n\n---\n\n'.join(partes)


# Comprobación rápida: invocamos la herramienta directamente para verificar que devuelve fragmentos con su cita correspondiente.
print('Herramienta definida:', buscar_en_base_conocimiento.name)
test = buscar_en_base_conocimiento.invoke({'consulta': 'qué es la similitud coseno'})
print('\nTest rápido (primeros 300 chars):')
print(test[:300] + '...')

Herramienta definida: buscar_en_base_conocimiento

Test rápido (primeros 300 chars):
[Ingenieria_de_Prompts.pdf | pág. 2]
8. Chain of Thought y razonamiento
Chain of Thought (CoT) mejora el razonamiento pidiendo al modelo que piense paso a paso
antes de responder.

Usar frases como "Piensa paso a paso".

Mejora tareas matemáticas y lógicas.

Reduce errores en razonamiento complej...


### 5.2. *System prompt*

In [ ]:
# El system prompt se inyecta como SystemMessage al inicio de cada turno.
SYSTEM_PROMPT = (
    """ Eres un asistente experto en IA Generativa. Tienes acceso a los PDFs del curso que cubren: LLMs, RAG, LangGraph, embeddings, prompt engineering y agentes.
    Reglas:
    1. Antes de responder cualquier pregunta técnica, usa SIEMPRE la herramienta `buscar_en_base_conocimiento` para buscar en los PDFs.
    2. Cita siempre el documento y la página de donde sacas la información.
    3. Responde en español, de forma clara y directa.
    4. Si no encuentras la información en los PDFs, dilo y responde con lo que sabes,
    pero indicando que es conocimiento general.
    5. Recuerda lo que hemos hablado antes en la conversación y aprovéchalo."""
)

print('System Prompt listo')
print(f'Longitud: {len(SYSTEM_PROMPT)} caracteres')

System Prompt listo
Longitud: 615 caracteres


### 5.3. Modelo de lenguaje

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Modelo principal del agente.
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3
)

# Registramos la herramienta.
tools = [buscar_en_base_conocimiento]
llm_con_tools = llm.bind_tools(tools)

print(f'Modelo: {llm.model}')
print(f'Herramientas registradas: {[t.name for t in tools]}')

Modelo: gemini-2.5-flash
Herramientas registradas: ['buscar_en_base_conocimiento']


## 6. Construcción del grafo en LangGraph

### 6.1. Definición del estado
Para definir qué información va a fluir entre los nodos del grafo — en este caso, el historial de mensajes de la conversación.


In [10]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, SystemMessage


# Estado compartido entre nodos: solo la lista de mensajes acumulados.
class EstadoAgente(TypedDict):
    mensajes: Annotated[list[BaseMessage], add_messages]

### 6.2. Nodo del agente

Recibe el estado actual, antepone el `SystemMessage` con las instrucciones globales y llama al LLM. La respuesta puede ser:

- Una respuesta final en lenguaje natural (no contiene `tool_calls`).
- Una solicitud de invocación de herramienta (`tool_calls` poblado).

In [11]:
def nodo_agente(estado: EstadoAgente) -> dict:
    '''Genera la siguiente acción del agente: respuesta final o tool_call.'''
    # Inyectamos el system prompt al principio para que se aplique en cada turno.
    mensajes = [SystemMessage(content=SYSTEM_PROMPT)] + estado['mensajes']
    respuesta = llm_con_tools.invoke(mensajes)
    # Devolvemos solo el mensaje nuevo: add_messages se encargará de acumularlo.
    return {'mensajes': [respuesta]}

### 6.3. *Router* condicional (ciclo ReAct)

El *router* inspecciona el último mensaje generado por el agente y decide la transición:

- Si contiene `tool_calls` → enviamos el flujo al nodo `tools` para ejecutar la herramienta.
- En caso contrario → la respuesta es definitiva y terminamos el grafo (`END`).

Tras ejecutar la herramienta, el flujo vuelve siempre al nodo `agente`, cerrando así el bucle.

In [ ]:
def router_react(estado: EstadoAgente) -> Literal['tools', '__end__']:
    '''Decide si pasar a la ejecución de herramientas o cerrar el grafo.'''
    ultimo = estado['mensajes'][-1]
    if hasattr(ultimo, 'tool_calls') and ultimo.tool_calls:
        return 'tools'
    return '__end__'


# ToolMessage para que el agente pueda leerlos en la siguiente iteración.
nodo_tools = ToolNode(tools, messages_key='mensajes')

print('Nodos y router definidos')

Nodos y router definidos


### 6.4. Construcción y compilación del grafo

Una vez definidos los nodos y el *router*, ensamblamos el grafo conectándolos. El paso final es la compilación con `MemorySaver`, que actúa como *checkpointer* y persiste el estado tras cada nodo. Cada conversación queda identificada por un `thread_id`, lo que permite mantener varias sesiones independientes en paralelo.

In [18]:
# Construcción del grafo
grafo = StateGraph(EstadoAgente)

# Registramos los dos nodos
grafo.add_node('agente', nodo_agente)
grafo.add_node('tools',  nodo_tools)

# El flujo siempre empieza en el nodo del agente
grafo.add_edge(START, 'agente')

# Arista condicional: el router decide a dónde ir tras la respuesta del agente
grafo.add_conditional_edges(
    'agente',
    router_react,
    {'tools': 'tools', '__end__': END}
)

# Tras ejecutar herramientas siempre se vuelve al agente (bucle ReAct)
grafo.add_edge('tools', 'agente')

# Compilación con persistencia en memoria.
checkpointer = MemorySaver()
agente = grafo.compile(checkpointer=checkpointer)

print('Agente compilado correctamente')
print(agente.get_graph().draw_ascii())

Agente compilado correctamente
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +--------+          
          | agente |          
          +--------+          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## 7. Función de interacción con el agente

Para encapsular las llamadas al grafo se define una función auxiliar `chat()` que recibe la pregunta del usuario y un `thread_id`. Este último es la clave que `MemorySaver` utiliza para asociar mensajes a una misma conversación: usando el mismo `thread_id` en llamadas sucesivas, el agente mantiene la memoria; usando uno distinto, comienza una conversación limpia.

In [21]:
from langchain_core.messages import HumanMessage


def chat(pregunta: str, thread_id: str = 'sesion') -> str:
    '''Envía una pregunta al agente y devuelve la respuesta final.'''
    config = {'configurable': {'thread_id': thread_id}}
    resultado = agente.invoke(
        {'mensajes': [HumanMessage(content=pregunta)]},
        config=config
    )
    ultimo = resultado['mensajes'][-1]
    content = ultimo.content

    # Gemini puede devolver el content como lista de partes (dicts con 'text').
    # Lo aplanamos a string para tener siempre una respuesta legible.
    if isinstance(content, list):
        partes = []
        for p in content:
            if isinstance(p, dict):
                partes.append(p.get('text', ''))
            else:
                partes.append(str(p))
        return '\n'.join(t for t in partes if t)
    return content


print('Función de interacción lista')

Función de interacción lista


## 8. Casos de prueba


### Ejemplo 1 — ¿Qué es RAG y por qué es útil?

Pregunta directa sobre uno de los conceptos centrales del módulo. Esperamos que el agente recupere fragmentos de `RAG_langgraph.pdf` y elabore una explicación citando la fuente.

In [22]:
r1 = chat('¿Qué es RAG y por qué es útil?', thread_id='ejemplos')
print(r1)

RAG, o Retrieval-Augmented Generation, es una arquitectura que combina la recuperación de información con la generación de texto por parte de modelos de lenguaje grandes (LLMs).

Es útil porque aborda limitaciones fundamentales de los LLMs cuando se usan de forma aislada:

*   **Conocimiento estático:** Los LLMs tienen un conocimiento limitado hasta la fecha de su entrenamiento. RAG permite incorporar información actualizada o específica de un dominio (como documentación interna de una empresa, normativas recientes o catálogos de productos) que no estaba presente en los datos de entrenamiento del LLM.
*   **Alucinaciones:** Al basar la generación en información recuperada y específica, RAG puede ayudar a reducir las "alucinaciones" o invenciones de hechos por parte del LLM.

En resumen, RAG hace que los LLMs sean más precisos, actualizados y confiables para aplicaciones empresariales al permitirles acceder y utilizar información externa relevante.

**Referencia:** [RAG_langgraph.pdf | 

### Ejemplo 2 — ¿Qué es LangGraph y cómo se usa para construir agentes?

Pregunta orientada al *framework* sobre el que se construye el propio agente. Permite observar cómo el modelo combina información de varios fragmentos para construir una explicación coherente.

In [24]:
r2 = chat('¿Qué es LangGraph y cómo se usa para construir agentes?', thread_id='ejemplos')
print(r2)

LangGraph es una extensión de LangChain que permite organizar flujos de ejecución complejos como grafos. Se utiliza para construir agentes mediante la definición de nodos (que representan pasos como el razonamiento, la generación de texto o la recuperación de información) y estados (que representan la información compartida entre estos nodos).

El flujo de ejecución se modela como un grafo, donde las rutas condicionales determinan la secuencia de nodos a seguir. Esto permite crear arquitecturas de agentes sofisticadas, como las que integran RAG (Retrieval-Augmented Generation) para la recuperación de contexto.

**Referencia:** [Orquestacion_Despliegue_Agentes.pdf | pág. 1] y [RAG_langgraph.pdf | pág. 2]


### Ejemplo 3 — Similitud coseno en RAG

Concepto matemático fundamental para entender el funcionamiento de la búsqueda vectorial. Aparece en el material teórico del módulo.

In [26]:
r3 = chat('¿Qué es la similitud coseno y para qué sirve en RAG?', thread_id='ejemplos')
print(r3)

La similitud coseno es una métrica utilizada para medir la similitud entre dos vectores no nulos en un espacio multidimensional. En el contexto de RAG, sirve para determinar qué tan relevante es un documento o fragmento de texto (representado como un vector) con respecto a una consulta (también representada como un vector).

Cuando se implementa RAG, la consulta del usuario se convierte en un vector. Luego, este vector se compara con los vectores que representan los documentos en la base de conocimiento. La similitud coseno calcula el coseno del ángulo entre estos vectores. Un valor cercano a 1 indica una alta similitud (los vectores apuntan en direcciones muy parecidas), mientras que un valor cercano a 0 indica baja similitud.

Por lo tanto, la similitud coseno es fundamental en la fase de "Recuperación" de RAG para encontrar los documentos más pertinentes que luego se utilizarán para aumentar la generación del LLM.

Aunque la información específica sobre la similitud coseno y su apli

### Ejemplo 4 — Generación y uso de embeddings

Pregunta que cubre la otra mitad del proceso: cómo se construye el espacio vectorial sobre el que después actúa el *retriever*.

In [28]:
r4 = chat('¿Cómo se generan los embeddings y cómo se usan para buscar documentos?', thread_id='ejemplos')
print(r4)

Los embeddings se generan transformando texto (tokens) en vectores numéricos densos. Estos vectores capturan el significado semántico del texto, de modo que palabras o frases con significados similares tienen representaciones vectoriales cercanas en un espacio multidimensional.

Para buscar documentos, los embeddings se utilizan de la siguiente manera:

1.  **Indexación:** Los documentos de una base de conocimiento se dividen en fragmentos (chunks) y cada fragmento se convierte en un vector de embedding utilizando un modelo de embeddings. Estos vectores se almacenan y se indexan en una base de datos vectorial (como ChromaDB).
2.  **Consulta:** Cuando un usuario realiza una consulta, esta también se convierte en un vector de embedding utilizando el mismo modelo.
3.  **Búsqueda por similitud:** El vector de la consulta se compara con los vectores de los documentos almacenados. Se utilizan métricas como la similitud coseno para encontrar los vectores (y por lo tanto, los documentos) más c

### Ejemplo 5 — Memoria en agentes de LangGraph

Pregunta de carácter recursivo: el agente describe el mecanismo de memoria que él mismo está utilizando para mantener el contexto de esta conversación.

In [30]:
r5 = chat('¿Cómo funciona la memoria en un agente de LangGraph?', thread_id='ejemplos')
print(r5)

La memoria de conversación es crucial para que un agente de LangGraph funcione como un asistente conversacional, permitiéndole recordar e interactuar basándose en intercambios anteriores. Sin memoria, el agente trataría cada pregunta de forma aislada.

En LangGraph, la memoria se implementa para que el agente pueda recordar el historial de la conversación. Esto se logra típicamente incluyendo un campo para el "historial" dentro de la definición del estado del agente (por ejemplo, `AgentState` en el ejemplo de `RAG_langgraph.pdf`). Este historial almacena las interacciones pasadas, permitiendo al agente referirse a ellas en turnos de conversación posteriores.

Existen diferentes tipos de memoria disponibles en LangChain/LangGraph, cada uno con sus ventajas y limitaciones, que permiten gestionar esta persistencia de información entre turnos.

**Referencias:** [RAG_langgraph.pdf | pág. 2] y [Orquestacion_Despliegue_Agentes.pdf | pág. 3]


## 9. Interfaz de chat interactivo

Como cierre práctico del proyecto, se incluye una pequeña interfaz construida con `ipywidgets` que permite conversar con el agente directamente desde el notebook. Cada mensaje se envía con el botón *Enviar* (o pulsando `Enter`), y la conversación se mantiene en un único `thread_id`, por lo que el agente recuerda lo dicho a lo largo de la sesión.

In [33]:
import ipywidgets as widgets
from IPython.display import display

# Identificador del hilo de la conversación interactiva
THREAD_CHAT = 'chat-interactivo'
mensajes_html = []

# ── Componentes visuales ──────────────────────────────────────────────────────
historial = widgets.HTML(
    value='<p style="color:#aaa;font-style:italic;text-align:center;margin-top:20px">'
          'Realiza la primera pregunta para comenzar.</p>'
)
caja = widgets.Box(
    [historial],
    layout=widgets.Layout(
        height='400px', overflow_y='auto',
        border='1px solid #e0e0e0', border_radius='12px',
        padding='16px', background_color='#ffffff'
    )
)
entrada = widgets.Text(
    placeholder='Escribe tu pregunta...',
    layout=widgets.Layout(width='82%', height='36px')
)
boton = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='16%', height='36px')
)
estado = widgets.HTML(value='')


# ── Lógica de envío ───────────────────────────────────────────────────────────
def enviar(_):
    """Maneja el evento de envío: pinta la pregunta, llama al agente y pinta la respuesta."""
    pregunta = entrada.value.strip()
    if not pregunta:
        return

    entrada.value = ''
    estado.value = '<span style="color:#888;font-size:12px">Procesando...</span>'

    # Burbuja del usuario (alineada a la derecha)
    mensajes_html.append(
        f'<div style="display:flex;justify-content:flex-end;margin:8px 0">'
        f'<div style="background:#0084ff;color:white;padding:10px 14px;'
        f'border-radius:18px 18px 4px 18px;max-width:75%;font-size:13px;'
        f'line-height:1.4;word-wrap:break-word">{pregunta}</div></div>'
    )
    historial.value = ''.join(mensajes_html)

    # Invocación del agente con el thread_id de la conversación
    try:
        respuesta = chat(pregunta, thread_id=THREAD_CHAT)
        respuesta_fmt = respuesta.replace('\n', '<br>')
        color_burbuja = '#f0f0f0'
        color_texto = '#1c1c1e'
    except Exception as e:
        respuesta_fmt = f'Error: {e}'
        color_burbuja = '#ffe5e5'
        color_texto = '#c0392b'

    # Burbuja del agente (alineada a la izquierda)
    mensajes_html.append(
        f'<div style="display:flex;justify-content:flex-start;margin:8px 0">'
        f'<div style="background:{color_burbuja};color:{color_texto};padding:10px 14px;'
        f'border-radius:18px 18px 18px 4px;max-width:75%;font-size:13px;'
        f'line-height:1.4;word-wrap:break-word">{respuesta_fmt}</div></div>'
    )
    historial.value = ''.join(mensajes_html)
    estado.value = ''


# Asociamos la función al click del botón y al Enter sobre el campo de texto
boton.on_click(enviar)
entrada.on_submit(enviar)

# ── Composición final ─────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML('<div style="font-size:16px;font-weight:bold;margin-bottom:8px">'
                 'Asistente IA Generativa</div>'),
    caja,
    widgets.HBox([entrada, boton],
                 layout=widgets.Layout(margin='8px 0 0 0')),
    estado
], layout=widgets.Layout(width='700px')))

C:\Users\Tato\AppData\Local\Temp\ipykernel_83232\2983241526.py:76: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  entrada.on_submit(enviar)
